# 🎯 Priority Queues — Cola de Prioridad ADT e Implementaciones con Listas

## Programación III - Lic. en Sistemas

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/prog3-ls-fcad-uner/blob/main/cap09/01_PriorityQueue_ADT_Teoria.ipynb)

### 🎯 Objetivos de esta clase

Al finalizar este notebook, serás capaz de:

1. ✅ Definir el **ADT Cola de Prioridad** y sus operaciones: `add(k,v)`, `min()`, `remove_min()`, `is_empty()`, `len()`
2. ✅ Implementar **`UnsortedPriorityQueue`** con lista no ordenada — O(1) add, O(n) min
3. ✅ Implementar **`SortedPriorityQueue`** con lista ordenada — O(n) add, O(1) min
4. ✅ Comparar ambas implementaciones y sus trade-offs

### 📋 Contenido

📚 *Data Structures and Algorithms in Python* — Goodrich, Tamassia, Goldwasser (Cap. 9)  

| Sección | Tema | Tiempo |
|---------|------|--------|
| 9.1 | Priority Queue ADT | 30 min |
| 9.2.1 | PriorityQueueBase y _Item | 15 min |
| 9.2.2 | UnsortedPriorityQueue | 25 min |
| 9.2.3 | SortedPriorityQueue | 25 min |
| — | Benchmark comparativo | 15 min |


---

## 📌 9.1 — El ADT Cola de Prioridad

### ¿Qué es una Cola de Prioridad?

Una **cola de prioridad** (*priority queue*) es una colección de pares **(clave, valor)** donde:

- La **clave** representa la **prioridad** del elemento (menor clave → mayor prioridad)
- El **valor** es el dato asociado (puede ser cualquier objeto)
- La operación fundamental extrae el elemento con la **clave mínima**

### 🌍 Aplicaciones del mundo real

| Escenario | Clave (prioridad) | Valor |
|-----------|------------------|-------|
| 🏥 Sala de urgencias | Gravedad del caso (1=crítico) | Paciente |
| ✈️ Lista de espera de vuelo | Tiempo de llegada a lista | Pasajero |
| 💻 Scheduler de OS | Prioridad del proceso | Proceso |
| 🗺️ Algoritmo de Dijkstra | Distancia mínima tentativa | Vértice |
| 📊 Compresión Huffman | Frecuencia del carácter | Símbolo |

### 📐 Interfaz del ADT

| Operación | Descripción |
|-----------|-------------|
| `P.add(k, v)` | Inserta el par (k,v) en la cola |
| `P.min()` | Retorna (k,v) con clave mínima **sin eliminar** |
| `P.remove_min()` | **Elimina y retorna** (k,v) con clave mínima |
| `P.is_empty()` | True si la cola está vacía |
| `len(P)` | Número de elementos en la cola |

> **⚠️ Error:** `P.min()` y `P.remove_min()` lanzan `Empty` si la cola está vacía.

### 📊 Traza de Operaciones (Tabla 9.1 del libro)

| Operación | Retorna | Estado de P |
|-----------|---------|-------------|
| `P.add(5, 'A')` | — | {(5,A)} |
| `P.add(9, 'C')` | — | {(5,A),(9,C)} |
| `P.add(3, 'B')` | — | {(3,B),(5,A),(9,C)} |
| `P.add(7, 'D')` | — | {(3,B),(5,A),(7,D),(9,C)} |
| `P.min()` | `(3,'B')` | {(3,B),(5,A),(7,D),(9,C)} |
| `P.remove_min()` | `(3,'B')` | {(5,A),(7,D),(9,C)} |
| `P.remove_min()` | `(5,'A')` | {(7,D),(9,C)} |
| `len(P)` | `2` | {(7,D),(9,C)} |
| `P.add(1, 'F')` | — | {(1,F),(7,D),(9,C)} |
| `P.remove_min()` | `(1,'F')` | {(7,D),(9,C)} |

> **Nota sobre claves iguales:** Si dos elementos tienen la misma clave, la cola puede retornar
> cualquiera de ellos. No se garantiza orden FIFO para claves iguales.


---

## 📌 9.2.1 — La Clase Base `PriorityQueueBase`

Antes de implementar, definimos una **clase base abstracta** con:

- La clase anidada `_Item` que encapsula el par (clave, valor)
- El operador `__lt__` en `_Item` para comparaciones basadas en **clave**
- El método `is_empty()` implementado usando `len()`

### 🔑 Clase Anidada `_Item`

```python
class _Item:
    __slots__ = '_key', '_value'
    
    def __init__(self, k, v):
        self._key = k
        self._value = v
    
    def __lt__(self, other):
        return self._key < other._key   # Compara por CLAVE
```

El `__lt__` es fundamental: permite que funciones como `min()` comparen elementos
sin exponer la clave directamente.


In [ ]:
class Empty(Exception):
    """Error raised when accessing an element from an empty container."""
    pass


class PriorityQueueBase:
    """Abstract base class for a priority queue."""

    # ------------------------------- nested _Item class -------------------------------
    class _Item:
        """Lightweight composite to store priority queue items."""
        __slots__ = '_key', '_value'

        def __init__(self, k, v):
            self._key = k
            self._value = v

        def __lt__(self, other):
            """Compare items based on their keys."""
            return self._key < other._key

        def __repr__(self):
            return f'({self._key}, {self._value!r})'

    # ------------------------------- public behaviors -------------------------------
    def is_empty(self):
        """Return True if the priority queue is empty."""
        return len(self) == 0


# ── Demostración de _Item ──
item_a = PriorityQueueBase._Item(5, 'A')
item_b = PriorityQueueBase._Item(3, 'B')
item_c = PriorityQueueBase._Item(9, 'C')

print('Demostración de _Item:')
print(f'  item_a = {item_a}')
print(f'  item_b = {item_b}')
print(f'  item_b < item_a? → {item_b < item_a}')          # True (3 < 5)
print(f'  min(a, b, c) = {min(item_a, item_b, item_c)}')   # (3, 'B')


---

## 📌 9.2.2 — `UnsortedPriorityQueue`

### 🧠 Estrategia: Lazy (Perezosa)

- **`add(k, v)`** → Inserta al **final** de la lista → **O(1)**
- **`min()` / `remove_min()`** → Recorre toda la lista buscando el mínimo → **O(n)**

La lista interna **no mantiene ningún orden**; el trabajo se pospone hasta la extracción.

### 📊 Complejidad

| Operación | Complejidad |
|-----------|-------------|
| `len()` | O(1) |
| `is_empty()` | O(1) |
| `add(k, v)` | **O(1)** |
| `min()` | **O(n)** |
| `remove_min()` | **O(n)** |


In [ ]:
class UnsortedPriorityQueue(PriorityQueueBase):
    """A min-oriented priority queue implemented with an unsorted list."""

    def __init__(self):
        """Create a new, empty priority queue."""
        self._data = []

    def __len__(self):
        return len(self._data)

    def _find_min(self):
        """Return index of item with minimum key. O(n) time."""
        if self.is_empty():
            raise Empty('Priority queue is empty')
        small_idx = 0
        for idx in range(1, len(self._data)):
            if self._data[idx] < self._data[small_idx]:
                small_idx = idx
        return small_idx

    def add(self, key, value):
        """Add a key-value pair. O(1) amortized time."""
        self._data.append(self._Item(key, value))

    def min(self):
        """Return but do not remove (k,v) tuple with minimum key. O(n)."""
        idx = self._find_min()
        item = self._data[idx]
        return (item._key, item._value)

    def remove_min(self):
        """Remove and return (k,v) tuple with minimum key. O(n)."""
        idx = self._find_min()
        item = self._data.pop(idx)
        return (item._key, item._value)

    def __repr__(self):
        return f'UnsortedPQ{list(self._data)}'


# ── Traza de la Tabla 9.1 ──
print('=== UnsortedPriorityQueue — Traza Tabla 9.1 ===')
P = UnsortedPriorityQueue()
P.add(5, 'A'); print(f'add(5,A)       → {P}')
P.add(9, 'C'); print(f'add(9,C)       → {P}')
P.add(3, 'B'); print(f'add(3,B)       → {P}')
P.add(7, 'D'); print(f'add(7,D)       → {P}')
print(f'min()          → {P.min()}')
print(f'remove_min()   → {P.remove_min()}')
print(f'remove_min()   → {P.remove_min()}')
print(f'len(P)         → {len(P)}')
P.add(1, 'F'); print(f'add(1,F)       → {P}')
print(f'remove_min()   → {P.remove_min()}')


---

## 📌 9.2.3 — `SortedPriorityQueue`

### 🧠 Estrategia: Eager (Ansiosa)

- **`add(k, v)`** → Inserta en la posición correcta para mantener orden → **O(n)**
- **`min()` / `remove_min()`** → El mínimo está siempre al **frente** → **O(1)**

El trabajo se hace al insertar para que la extracción sea barata.

### 📊 Complejidad

| Operación | Complejidad |
|-----------|-------------|
| `len()` | O(1) |
| `is_empty()` | O(1) |
| `add(k, v)` | **O(n)** |
| `min()` | **O(1)** |
| `remove_min()` | **O(1)** |

### 💡 La Dualidad

| | Unsorted PQ | Sorted PQ | Heap PQ |
|-|:-----------:|:---------:|:-------:|
| `add` | **O(1)** | O(n) | O(log n) |
| `min` / `remove_min` | O(n) | **O(1)** | O(log n) / O(1) |


In [ ]:
class SortedPriorityQueue(PriorityQueueBase):
    """A min-oriented priority queue implemented with a sorted list."""

    def __init__(self):
        """Create a new, empty priority queue."""
        self._data = []   # Ordenada de menor a mayor clave

    def __len__(self):
        return len(self._data)

    def add(self, key, value):
        """Add a key-value pair maintaining sorted order. O(n)."""
        new_item = self._Item(key, value)
        # Recorrer de derecha a izquierda para encontrar punto de inserción
        idx = len(self._data)
        while idx > 0 and new_item < self._data[idx - 1]:
            idx -= 1
        self._data.insert(idx, new_item)

    def min(self):
        """Return but do not remove (k,v) with minimum key. O(1)."""
        if self.is_empty():
            raise Empty('Priority queue is empty')
        item = self._data[0]
        return (item._key, item._value)

    def remove_min(self):
        """Remove and return (k,v) with minimum key. O(1) amortized."""
        if self.is_empty():
            raise Empty('Priority queue is empty')
        item = self._data.pop(0)
        return (item._key, item._value)

    def __repr__(self):
        return f'SortedPQ{list(self._data)}'


# ── Demostración ──
print('=== SortedPriorityQueue — Traza ===')
Q = SortedPriorityQueue()
Q.add(5, 'A'); print(f'add(5,A)       → {Q}')
Q.add(9, 'C'); print(f'add(9,C)       → {Q}')
Q.add(3, 'B'); print(f'add(3,B)       → {Q}')  # Se inserta al frente
Q.add(7, 'D'); print(f'add(7,D)       → {Q}')  # Entre (5,A) y (9,C)
print(f'min()          → {Q.min()}')
print(f'remove_min()   → {Q.remove_min()}')
print(f'remove_min()   → {Q.remove_min()}')
print(f'len(Q)         → {len(Q)}')


---

## 📊 9.2.4 — Benchmark Comparativo

Medimos empíricamente las diferencias de rendimiento entre ambas implementaciones.


In [ ]:
import time
import random

def benchmark_pq(pq_class, n, seed=42):
    """Mide tiempo de n inserciones + n extracciones."""
    random.seed(seed)
    keys = [random.randint(1, 10000) for _ in range(n)]
    pq = pq_class()

    t0 = time.perf_counter()
    for k in keys:
        pq.add(k, f'v_{k}')
    t_add = time.perf_counter() - t0

    t0 = time.perf_counter()
    result = []
    while not pq.is_empty():
        result.append(pq.remove_min())
    t_remove = time.perf_counter() - t0

    is_sorted = all(result[i][0] <= result[i+1][0] for i in range(len(result)-1))
    return t_add * 1000, t_remove * 1000, is_sorted


print(f'{"N":>6} | {"Impl":25} | {"add (ms)":>10} | {"remove (ms)":>12} | {"Ordenado":>9}')
print('-' * 75)

for N in [500, 1000, 2000]:
    for cls in [UnsortedPriorityQueue, SortedPriorityQueue]:
        t_a, t_r, ok = benchmark_pq(cls, N)
        print(f'{N:>6} | {cls.__name__:25} | {t_a:>10.2f} | {t_r:>12.2f} | {str(ok):>9}')
    print()

print('💡 UnsortedPQ: add O(1) → rápido   |  remove O(n) → lento')
print('   SortedPQ:   add O(n) → lento    |  remove O(1) → rápido')
print('   ⚡ Heap (cap 9.3): add O(logN)  |  remove O(logN) → balance óptimo')


---

## 📝 Resumen

### Conceptos clave

| Concepto | Resumen |
|----------|---------|
| **ADT Cola de Prioridad** | Colección de pares (clave, valor); extrae siempre el de menor clave |
| **`_Item`** | Clase anidada que encapsula (k, v) con comparación por clave |
| **`UnsortedPriorityQueue`** | add O(1), min/remove_min O(n) — estrategia lazy |
| **`SortedPriorityQueue`** | add O(n), min/remove_min O(1) — estrategia eager |

### ¿Qué viene en Notebook 2?

El **heap** resuelve el balance: `add` en O(log n), `min` en O(1), `remove_min` en O(log n).  
Veremos la `HeapPriorityQueue` y el módulo `heapq` de Python.

---

🔗 **Referencia:** Goodrich et al., Cap. 9.1–9.2

---

![Creative Commons](https://mirrors.creativecommons.org/presskit/buttons/80x15/png/by-sa.png)

© 2026 Cátedra Programación III – Lic. Sistemas – FCAD/UNER

This notebook is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License (CC BY-SA 4.0)

<https://creativecommons.org/licenses/by-sa/4.0/>